# Feature Engineering Pipeline (Revised)

For a detailed log of pipeline revisions, see `SourceOfTruth.md#5-pipeline-revision-log`.


In [2]:
from pathlib import Path
Path('data').mkdir(exist_ok=True)

import pandas as pd
import numpy as np
import gc
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [3]:
gc.collect()
df_Master = pd.read_parquet('data/df_Master_Final.parquet')
gc.collect()
# Validate: check for audit trail column from DataCleaning-Rev
if 'original_amount_header' in df_Master.columns:
    print("[OK]Audit trail column 'original_amount_header' present")
    gc.collect()
else:
    gc.collect()
    print("[ERROR] 'original_amount_header' not found. Ensure DataCleaning-Rev was run.")
    print(f"Columns available: {', '.join(df_Master.columns.tolist()[:10])}...")

[OK]Audit trail column 'original_amount_header' present


In [4]:
gc.collect()

0

## Temporal Feature Engineering

Derives time-based features from `created_at`: hour, month, day_name, weekend flag, transaction period, etc.

In [5]:
df_Master['hour'] = df_Master['created_at'].dt.hour
gc.collect()
df_Master['month'] = df_Master['created_at'].dt.month
gc.collect()
df_Master['day_name'] = df_Master['created_at'].dt.day_name()
gc.collect()
df_Master['is_weekend'] = np.where(
    df_Master['day_name'].isin(['Saturday', 'Sunday']),
    'Weekend',
    'Weekday'
)
gc.collect()
df_Master['member_status'] = np.where(df_Master['user_id'].notna(), 'Member', 'Guest')
gc.collect()
df_Master['is_voucher_used'] = np.where(
    df_Master['voucher_id'].notna(),
    'Voucher',
    'No Voucher'
)
gc.collect()
df_Master['month_name'] = (
    df_Master['created_at']
    .dt.month_name()
)
gc.collect()
df_Master['transaction_period'] = np.select(
    [
        df_Master['hour'].between(5, 10),
        df_Master['hour'].between(11, 15),
        df_Master['hour'].between(16, 19),
        df_Master['hour'].between(20, 23)
    ],
    [
        'Morning',
        'Afternoon',
        'Evening',
        'Night'
    ],
    default='Late Night'
)
df_Master['discount_ratio'] = (
    df_Master['discount_applied'] / (df_Master['original_amount'] + 1e-6)
)
gc.collect()

# FIX #4: Add boolean-encoded columns for modeling
df_Master['is_weekend_bool'] = np.where(
    df_Master['is_weekend'] == 'Weekend', 1, 0
).astype('int8')
df_Master['is_voucher_used_bool'] = np.where(
    df_Master['is_voucher_used'] == 'Voucher', 1, 0
).astype('int8')
gc.collect()

0

In [6]:
print(df_Master.columns.tolist())

['transaction_id', 'item_id', 'quantity', 'unit_price', 'subtotal', 'item_name', 'category_x', 'price', 'is_seasonal', 'store_id', 'payment_method_id', 'voucher_id', 'user_id', 'original_amount', 'discount_applied', 'final_amount', 'Non-Member', 'Member', 'original_amount_header', 'voucher_code', 'discount_type', 'discount_value', 'valid_from', 'valid_to', 'calculated_discount', 'gender', 'birthdate', 'registered_at', 'store_name', 'street', 'postal_code', 'city', 'state', 'latitude', 'longitude', 'created_at', 'method_id', 'method_name', 'category_y', 'hour', 'month', 'day_name', 'is_weekend', 'member_status', 'is_voucher_used', 'month_name', 'transaction_period', 'discount_ratio', 'is_weekend_bool', 'is_voucher_used_bool']


In [7]:
gc.collect()

0

In [8]:
gc.collect()
df_Master = df_Master.rename(columns={
    'category_x': 'menu_category',
    'category_y': 'payment_category'
})
gc.collect()

24

In [9]:
gc.collect()
df_Master = df_Master.drop(
    columns=['method_id', 'Non-Member', 'Member'],
    errors='ignore'
)
gc.collect()

0

## Transaction-Level Aggregation

In [10]:
gc.collect()
df_transaction_features = (
    df_Master
    .groupby('transaction_id')
    .agg({
        'quantity': 'sum',           
        'final_amount': 'max',       
        'discount_applied': 'max',    
        'is_weekend_bool': 'max',     
        'is_voucher_used_bool': 'max',
        'hour': 'max',               
        'month_name': 'max',         
        'day_name': 'max',            
        'city': 'max',               
        'method_name': 'max',         
        'payment_category': 'max',    
        'member_status': 'max',       
        'created_at': 'max',          
        'user_id': 'max',             
        'transaction_period': 'max'   
    })
    .reset_index()
    .rename(columns={
        'quantity': 'basket_size'
    })
)

# Add item_count: number of distinct items per transaction
gc.collect()
item_counts = df_Master.groupby('transaction_id')['item_id'].nunique().reset_index()
gc.collect()
item_counts.columns = ['transaction_id', 'item_count']
df_transaction_features = df_transaction_features.merge(item_counts, on='transaction_id', how='left')
gc.collect()
print(f"Transaction features: {len(df_transaction_features):,} rows, {len(df_transaction_features.columns)} columns")

Transaction features: 14,623,691 rows, 17 columns


In [11]:
gc.collect()

0

## RFM Analysis (Member Customers Only)

Recency, Frequency, Monetary analysis for the 2.2M registered members.

In [12]:
gc.collect()
snapshot_date = (
    df_transaction_features['created_at'].max()
    + pd.Timedelta(days=1)
)
gc.collect()

gc.collect()
rfm_table = (
    df_transaction_features[
        df_transaction_features['user_id'].notna()
    ]
    .groupby('user_id')
    .agg(
        Recency=('created_at', lambda x: (snapshot_date - x.max()).days),
        Frequency=('transaction_id', 'count'),
        Monetary=('final_amount', 'sum')
    )
    .reset_index()
)
gc.collect()

rfm_table['is_repeat_customer'] = np.where(
    rfm_table['Frequency'] > 1,
    'Repeat Customer',
    'One-Time Customer'
)
gc.collect()

print(f"RFM table: {len(rfm_table):,} customers, {len(rfm_table.columns)} columns")
print(f"Snapshot date: {snapshot_date}")
print(rfm_table[['Recency', 'Frequency', 'Monetary']].describe())

RFM table: 2,196,257 customers, 5 columns
Snapshot date: 2025-07-01 19:59:39
            Recency     Frequency      Monetary
count  2.196257e+06  2.196257e+06  2.196257e+06
mean   8.655728e+01  3.329694e+00  1.030858e+02
std    6.914141e+01  2.784946e+00  9.081546e+01
min    1.000000e+00  1.000000e+00  0.000000e+00
25%    3.100000e+01  1.000000e+00  3.900000e+01
50%    7.000000e+01  2.000000e+00  7.500000e+01
75%    1.270000e+02  4.000000e+00  1.380000e+02
max    5.220000e+02  4.000000e+01  1.348420e+03


In [13]:
gc.collect()

0

## FIX #: Scaled Features for K-Means Readiness

**Problem**: Raw RFM features have vastly different scales:
- Recency: 0–730 days
- Frequency: 1–? transactions (power-law distribution)
- Monetary: Rp30–? total spend

K-Means uses **Euclidean distance**. Without scaling, Monetary (large numbers) dominates the distance calculation, making Recency and Frequency nearly irrelevant to the clustering.

**Fix**: Apply StandardScaler (z-score normalization) to create `RFM_Scaled_*` columns. Also include a MinMaxScaler alternative (commented) for teams that prefer 0-1 scaling.

Both scaled and unscaled versions are preserved in the output so downstream notebooks can choose.

In [14]:
# Select RFM columns for scaling
gc.collect()
rfm_cols = ['Recency', 'Frequency', 'Monetary']

# Option 1: StandardScaler (z-score, mean=0, std=1)
# Preferred for K-Means when data has outliers (it's more robust)
gc.collect()
scaler_std = StandardScaler()
gc.collect()
rfm_scaled_std = scaler_std.fit_transform(rfm_table[rfm_cols])

gc.collect()
rfm_table['RFM_Scaled_Recency'] = rfm_scaled_std[:, 0]
gc.collect()
rfm_table['RFM_Scaled_Frequency'] = rfm_scaled_std[:, 1]
gc.collect()
rfm_table['RFM_Scaled_Monetary'] = rfm_scaled_std[:, 2]

# Option 2: MinMaxScaler (0-1 range)
# Alternative if you want bounded features
# scaler_mm = MinMaxScaler()
# rfm_scaled_mm = scaler_mm.fit_transform(rfm_table[rfm_cols])
# rfm_table['RFM_MM_Recency'] = rfm_scaled_mm[:, 0]
# rfm_table['RFM_MM_Frequency'] = rfm_scaled_mm[:, 1]
# rfm_table['RFM_MM_Monetary'] = rfm_scaled_mm[:, 2]

print("Scaled RFM features added.")
print(rfm_table[[c for c in rfm_table.columns if 'Scaled' in c]].describe().round(3))

Scaled RFM features added.
       RFM_Scaled_Recency  RFM_Scaled_Frequency  RFM_Scaled_Monetary
count         2196257.000           2196257.000          2196257.000
mean               -0.000                -0.000                0.000
std                 1.000                 1.000                1.000
min                -1.237                -0.837               -1.135
25%                -0.804                -0.837               -0.706
50%                -0.239                -0.477               -0.309
75%                 0.585                 0.241                0.384
max                 6.298                13.167               13.713


In [15]:
gc.collect()
snapshot_date = (
    df_transaction_features['created_at'].max()
    + pd.Timedelta(days=1)
)
gc.collect()

# Filter guest transactions
guest_tx = df_transaction_features[
    df_transaction_features['user_id'].isna()
].copy()
gc.collect()

# GROUP BY TRANSACTION_ID (bukan user_id)
rfm_guest = (
    guest_tx
    .groupby('transaction_id')
    .agg(
        Recency=('created_at', lambda x: (snapshot_date - x.max()).days),
        Frequency=('transaction_id', 'count'),    # <- akan selalu 1
        Monetary=('final_amount', 'sum')
    )
    .reset_index()
)
gc.collect()

# Karena Frequency selalu 1, is_repeat_customer juga selalu "One-Time"
rfm_guest['is_repeat_customer'] = 'One-Time Customer'

print(f"RFM Guest (per transaction): {len(rfm_guest):,} transaksi")
print(rfm_guest[['Recency', 'Monetary']].describe())

RFM Guest (per transaction): 7,310,827 transaksi
            Recency      Monetary
count  7.310827e+06  7.310827e+06
mean   5.321486e+02  3.093495e+01
std    1.305741e+02  1.571464e+01
min    1.000000e+00  0.000000e+00
25%    4.420000e+02  1.800000e+01
50%    5.460000e+02  2.850000e+01
75%    6.400000e+02  4.200000e+01
max    7.310000e+02  7.150000e+01


In [16]:
print(rfm_guest[['Recency', 'Frequency', 'Monetary']].describe())

            Recency  Frequency      Monetary
count  7.310827e+06  7310827.0  7.310827e+06
mean   5.321486e+02        1.0  3.093495e+01
std    1.305741e+02        0.0  1.571464e+01
min    1.000000e+00        1.0  0.000000e+00
25%    4.420000e+02        1.0  1.800000e+01
50%    5.460000e+02        1.0  2.850000e+01
75%    6.400000e+02        1.0  4.200000e+01
max    7.310000e+02        1.0  7.150000e+01


In [17]:
gc.collect()

0

## Temporal Train/Test Split for XGBoost Readiness

In [18]:
# Sort by timestamp to ensure correct temporal ordering
gc.collect()
df_transaction_features = df_transaction_features.sort_values('created_at').reset_index(drop=True)

# Find 80/20 temporal cutoff
gc.collect()
cutoff_idx = int(len(df_transaction_features) * 0.8)
gc.collect()
cutoff_date = df_transaction_features.iloc[cutoff_idx]['created_at']

# Temporal split: train on OLDEST 80%, test on NEWEST 20%
gc.collect()
df_train = df_transaction_features[df_transaction_features['created_at'] < cutoff_date].copy()
gc.collect()
df_test = df_transaction_features[df_transaction_features['created_at'] >= cutoff_date].copy()

print(f"Temporal split (80/20 by date):")
gc.collect()
print(f"  Full dataset: {len(df_transaction_features):,} transactions")
gc.collect()
print(f"  Train set:    {len(df_train):,} ({100*len(df_train)/len(df_transaction_features):.1f}%)")
gc.collect()
print(f"  Test set:     {len(df_test):,} ({100*len(df_test)/len(df_transaction_features):.1f}%)")
gc.collect()
print(f"  Cutoff date:  {cutoff_date}")
gc.collect()
print(f"  Train range:  {df_train['created_at'].min()} to {df_train['created_at'].max()}")
print(f"  Test range:   {df_test['created_at'].min()} to {df_test['created_at'].max()}")
gc.collect()

Temporal split (80/20 by date):
  Full dataset: 14,623,691 transactions
  Train set:    11,698,952 (80.0%)
  Test set:     2,924,739 (20.0%)
  Cutoff date:  2025-02-04 14:33:49
  Train range:  2023-07-01 07:00:00 to 2025-02-04 14:33:48
  Test range:   2025-02-04 14:33:49 to 2025-06-30 19:59:39


0

In [19]:
gc.collect()

0

## Save All Outputs

Four Parquet files are produced for downstream consumption:
1. `df_Master_FE.parquet` — Full item-level master with features (26.8M rows, for detailed analysis)
2. `df_transaction_features.parquet` — Transaction-level features (14.6M rows, for XGBoost)
3. `df_rfm.parquet` — RFM metrics + scaled features (for K-Means)
4. `df_train.parquet` — Temporal train split (for XGBoost without leakage)
5. `df_test.parquet` — Temporal test split (for XGBoost evaluation)

In [20]:
# Save full master with features
df_Master.to_parquet(
    'data/df_Master_FE.parquet',
    index=False
)
gc.collect()

# Save transaction-level features (for XGBoost)
df_transaction_features.to_parquet(
    'data/df_transaction_features.parquet',
    index=False
)
gc.collect()

# Save RFM with scaled features (for K-Means)
rfm_table.to_parquet(
    'data/df_rfm_member.parquet',
    index=False
)
gc.collect()

# Save temporal train/test splits (for XGBoost without leakage)
df_train.to_parquet('data/df_train.parquet', index=False)
df_test.to_parquet('data/df_test.parquet', index=False)
gc.collect()

print("All outputs saved successfully.")

All outputs saved successfully.


In [21]:
gc.collect()

0

## Apriori / Market Basket Analysis — Basket Preparation Pipeline

This section builds a binary transaction×item matrix optimized for Association Rule Mining (Apriori) using `mlxtend.frequent_patterns`.

### Pipeline Steps
1. **Load & Cast**: Load item-level and menu data; cast to efficient dtypes (`category`, `int8`) to minimize memory.
2. **Rare-Item Filter**: Drop items appearing in < 0.1% of all transactions (reduces noise and matrix width).
3. **Sparse Pivot**: Group by `transaction_id` × `item_name` → pivot to wide binary matrix.
4. **Edge-Case Removal**: Drop single-item baskets and bulk orders (>30 unique items).
5. **Save**: Persist as `df_basket_apriori.parquet` with shape and memory diagnostics.

In [22]:
import pandas as pd
import numpy as np
import gc

# --- 1. Load item-level transactional data ---
gc.collect()
df_items = pd.read_parquet('data/df_transaction_items_cleaned.parquet')

# --- 2. Load menu catalog for standardised item names ---
df_menu = pd.read_parquet('data/df_menu_cleaned.parquet')
gc.collect()

# --- 3. Cast to efficient dtypes ---
df_items['transaction_id'] = df_items['transaction_id'].astype('str')
df_items['item_id'] = df_items['item_id'].astype('int8')
df_items['quantity'] = df_items['quantity'].astype('int8')
gc.collect()

df_menu['item_id'] = df_menu['item_id'].astype('int8')
df_menu['item_name'] = df_menu['item_name'].astype('category')
df_menu['category'] = df_menu['category'].astype('category')
gc.collect()

# --- 4. Merge items with menu to attach standardised item names ---
df_items = df_items.merge(df_menu[['item_id', 'item_name']], on='item_id', how='left')
df_items['item_name'] = df_items['item_name'].cat.add_categories('unknown_item').fillna('unknown_item')
gc.collect()

print(f"Items loaded: {len(df_items):,} rows")
print(f"Unique transactions: {df_items['transaction_id'].nunique():,}")
print(f"Unique items: {df_items['item_name'].nunique():,}")

Items loaded: 26,885,688 rows
Unique transactions: 14,623,691
Unique items: 8


In [23]:
# --- 5. Compute global item support ---
total_transactions = df_items['transaction_id'].nunique()
MIN_SUPPORT_THRESHOLD = 0.001  # 0.1%

gc.collect()
item_frequency = (
    df_items.groupby('item_name', observed=True)['transaction_id']
    .nunique()
    .reset_index()
    .rename(columns={'transaction_id': 'transaction_count'})
)
item_frequency['global_support'] = item_frequency['transaction_count'] / total_transactions
gc.collect()

# --- 6. Identify and remove rare items ---
rare_items = item_frequency[item_frequency['global_support'] < MIN_SUPPORT_THRESHOLD]['item_name']
print(f"Rare items (< {MIN_SUPPORT_THRESHOLD:.1%} support): {len(rare_items):,} of {len(item_frequency):,} unique items")

df_items_filtered = df_items[~df_items['item_name'].isin(rare_items)].copy()
gc.collect()
print(f"Rows retained after filtering: {len(df_items_filtered):,} ({len(df_items_filtered)/len(df_items):.1%} of {len(df_items):,})")

# Free memory
del df_items, item_frequency, rare_items
gc.collect()

Rare items (< 0.1% support): 0 of 8 unique items
Rows retained after filtering: 26,885,688 (100.0% of 26,885,688)


0

In [24]:
# --- 7. Group by transaction x item and sum quantities ---
gc.collect()
basket_grouped = (
    df_items_filtered
    .groupby(['transaction_id', 'item_name'], observed=True, sort=False)
    ['quantity']
    .sum()
    .reset_index()
)
gc.collect()

gc.collect()
print(f"Grouped rows: {len(basket_grouped):,}")

# --- 8. Pivot to wide binary matrix ---
print("Pivoting to basket matrix (this may take a moment)...")
gc.collect()
basket_matrix = (
    basket_grouped
    .pivot_table(
        index='transaction_id',
        columns='item_name',
        values='quantity',
        fill_value=0,
        observed=True
    )
)
gc.collect()

# Convert to binary: 1 if purchased, 0 otherwise
basket_matrix = basket_matrix.astype(bool).astype('int8')
gc.collect()

print(f"Basket matrix shape: {basket_matrix.shape[0]:,} transactions x {basket_matrix.shape[1]:,} items")
mem_mb = basket_matrix.memory_usage(deep=True).sum() / 1024**2
print(f"Memory footprint: {mem_mb:.2f} MB")

del basket_grouped
gc.collect()

Grouped rows: 26,885,688
Pivoting to basket matrix (this may take a moment)...
Basket matrix shape: 14,623,691 transactions x 8 items
Memory footprint: 1297.00 MB


0

In [25]:
# --- 9. Remove single-item transactions ---
gc.collect()
row_sums = basket_matrix.sum(axis=1)
gc.collect()
single_item_mask = row_sums == 1
gc.collect()
n_single = single_item_mask.sum()
print(f"Single-item transactions (removed): {n_single:,} ({n_single/len(basket_matrix)*100:.1f}%)")

basket_matrix = basket_matrix[~single_item_mask]
gc.collect()

# --- 10. Remove bulk / catering orders (>30 unique items) ---
bulk_mask = row_sums[~single_item_mask] > 30
n_bulk = bulk_mask.sum()
print(f"Bulk transactions >30 items (removed): {n_bulk:,} ({n_bulk/len(basket_matrix)*100:.1f}%)")

basket_matrix = basket_matrix[~bulk_mask]
gc.collect()

print(f"Final basket matrix: {basket_matrix.shape[0]:,} transactions x {basket_matrix.shape[1]:,} items")

Single-item transactions (removed): 5,559,022 (38.0%)
Bulk transactions >30 items (removed): 0 (0.0%)
Final basket matrix: 9,064,669 transactions x 8 items


In [26]:
# --- 11. Save to Parquet ---
basket_matrix.to_parquet('data/df_basket_apriori.parquet')
gc.collect()

# --- 12. Final summary ---
gc.collect()
n_trans, n_items = basket_matrix.shape
gc.collect()
total_cells = n_trans * n_items
gc.collect()
filled_cells = basket_matrix.values.sum()
gc.collect()
sparsity = 1 - (filled_cells / total_cells)
gc.collect()
mem_mb = basket_matrix.memory_usage(deep=True).sum() / 1024**2

print("=" * 60)
print("   APRIORI BASKET PREPARATION -- COMPLETE")
print("=" * 60)
print(f"  Output file:     df_basket_apriori.parquet")
print(f"  Transactions:    {n_trans:,}")
print(f"  Items (columns): {n_items:,}")
print(f"  Sparsity:        {sparsity:.4%}")
print(f"  Memory:          {mem_mb:.2f} MB")
print("=" * 60)
print("Ready for Apriori (mlxtend.frequent_patterns).")

   APRIORI BASKET PREPARATION -- COMPLETE
  Output file:     df_basket_apriori.parquet
  Transactions:    9,064,669
  Items (columns): 8
  Sparsity:        70.5909%
  Memory:          803.96 MB
Ready for Apriori (mlxtend.frequent_patterns).
